# Peak Shaving: Control the heat pump

You control the house heat pump with a single **forcing signal**:

- return `1` &rarr; force the heat pump **OFF**
- return `0` &rarr; let it run normally (the thermostat decides)

Run the setup cell once, then edit `forcing_signal` and re-run the exercise
cell as often as you like. Watch the heat pump power and the indoor
temperature respond, and try to keep the pump off during the shaded
congestion windows **without** discomforting the occupants.

In [ ]:
# --- Setup: run this once ---
%matplotlib inline
import matplotlib.pyplot as plt
import cosimulate as cs

# Point the co-sim at the FMU shipped in this repo
cs.FMU_PATH = "Standalone_House.fmu"

ctx = cs.build_fmu_context(cs.FMU_PATH)
print("FMU ready. Congestion windows:", cs.CONGESTION_WINDOWS)

## Your turn: edit `forcing_signal`

`t_hours` is the time of day (0&ndash;24). `t_indoor_c` is the indoor
temperature if you want to react to the house instead of the clock.

Ideas: shed both windows, add a comfort floor, or close the loop on
temperature. Then run the cell.

In [ ]:
# ===== EDIT ME =====
def forcing_signal(t_hours, t_indoor_c=None):
    if 17 <= t_hours < 21:
        return 1        # force heat pump OFF
    return 0            # let it run
# ===================

def forcing_policy(t_seconds, tindoor_c):
    return forcing_signal((t_seconds / 3600.0) % 24.0, tindoor_c)

base   = cs.simulate(cs.baseline_policy, ctx, label="baseline")
active = cs.simulate(forcing_policy,     ctx, label="yours")

cs.summarize("Baseline", base)
cs.summarize("Your signal", active)

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(base["t_h"],   base["p_hp"],   label="Baseline", color="tab:red")
ax[0].plot(active["t_h"], active["p_hp"], label="Yours",    color="tab:blue")
ax[0].set_ylabel("Heat pump power [W]"); ax[0].legend(loc="upper right")
ax[0].set_title("Heat pump response to your forcing signal")
ax[1].plot(base["t_h"],   base["t_in_c"],   color="tab:red")
ax[1].plot(active["t_h"], active["t_in_c"], color="tab:blue")
ax[1].set_ylabel("Indoor temp [degC]"); ax[1].set_xlabel("Time [h]")
for a in ax:
    for lo, hi in cs.CONGESTION_WINDOWS:
        a.axvspan(lo, hi, color="orange", alpha=0.12)
plt.show()

## Challenge

Get the *energy in the congestion windows* down while keeping
*discomfort hours* near zero. Watch out for the **rebound**: shedding load
during a window often just makes the pump run harder right after it.